In [1]:
import os
import math
import time
import inspect
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import sentencepiece as spm

In [2]:
spp = spm.SentencePieceProcessor(model_file='../med_fine_sp.model')
vocab_size = spp.get_piece_size()
print(vocab_size)

50257


In [3]:
@dataclass
class config:
    block_size: int = 1024
    vocab_size: int = spp.get_piece_size()
    n_layer: int = 14
    n_head: int = 12
    n_kv_heads: int = 4
    n_embd: int = 768
    dropout: float = 0.3

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight


def precompute_rope_frequencies(head_dim, max_seq_len, theta=10000.0):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_seq_len)
    freqs = torch.einsum('i,j->ij', t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    half = x.shape[-1] // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

In [5]:
class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_heads
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.n_kv_groups = self.n_head // self.n_kv_head

        self.q_proj = nn.Linear(config.n_embd, self.n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.n_embd, self.n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.n_embd, self.n_kv_head * self.head_dim, bias=False)
        self.out_proj = nn.Linear(self.n_head * self.head_dim, config.n_embd, bias=False)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        k = k.repeat_interleave(self.n_kv_groups, dim=1)
        v = v.repeat_interleave(self.n_kv_groups, dim=1)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.attn_dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.out_proj(y))
        return y

In [6]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden_dim = int(8 * config.n_embd / 3)
        self.gate_proj = nn.Linear(config.n_embd, hidden_dim, bias=False)
        self.up_proj = nn.Linear(config.n_embd, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, config.n_embd, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

In [7]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_norm = RMSNorm(config.n_embd)
        self.attn = Attention(config)
        self.post_attn_norm = RMSNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.input_norm(x), cos, sin)
        x = x + self.mlp(self.post_attn_norm(x))
        return x

In [8]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        # Precompute RoPE frequencies
        head_dim = config.n_embd // config.n_head
        cos, sin = precompute_rope_frequencies(head_dim, config.block_size)
        self.register_buffer('rope_cos', cos)
        self.register_buffer('rope_sin', sin)

        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=RMSNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # Weight tying: input embedding = output head
        self.transformer.wte.weight = self.lm_head.weight

        # Init weights
        self.apply(self._init_weights)
        # Scaled init for residual projection weights (GPT-2 style)
        for pn, p in self.named_parameters():
            if pn.endswith('out_proj.weight') or pn.endswith('down_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

        print("number of parameters: %.2fM" % (self.get_num_params() / 1e6,))

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, RMSNorm):
            torch.nn.init.ones_(module.weight)

    def forward(self, idx, targets=None):
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"

        tok_emb = self.transformer.wte(idx)
        x = self.transformer.drop(tok_emb)

        cos = self.rope_cos[:t, :]
        sin = self.rope_sin[:t, :]

        for block in self.transformer.h:
            x = block(x, cos, sin)

        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None

        return logits, loss

    def crop_block_size(self, block_size):
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        # Recompute RoPE for new block size
        head_dim = self.config.n_embd // self.config.n_head
        cos, sin = precompute_rope_frequencies(head_dim, block_size)
        self.register_buffer('rope_cos', cos)
        self.register_buffer('rope_sin', sin)

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0},
        ]
        num_decay = sum(p.numel() for p in decay_params)
        num_nodecay = sum(p.numel() for p in nodecay_params)
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay:,} parameters")
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
        print(f"using fused AdamW: {use_fused}")
        return optimizer

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.n_layer, cfg.n_head, cfg.n_embd // cfg.n_head, cfg.block_size
        flops_per_token = 6 * N + 12 * L * H * Q * T
        flops_per_fwdbwd = flops_per_token * T
        flops_per_iter = flops_per_fwdbwd * fwdbwd_per_iter
        flops_achieved = flops_per_iter * (1.0 / dt)
        flops_promised = 312e12
        return flops_achieved / flops_promised

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [9]:
import torch; print(f'CUDA available: {torch.cuda.is_available()}'); print(f'Device name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None' }')

CUDA available: True
Device name: NVIDIA GeForce RTX 4050 Laptop GPU


In [10]:
# Training Loop with Tokenization
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Initialize model
cfg = config()
model = GPT(cfg)
model = model.to(device)

# Create optimizer
optimizer = model.configure_optimizers(
    weight_decay=0.01,
    learning_rate=1e-4,
    betas=(0.9, 0.95),
    device_type=device
)
print("✓ Optimizer configured")

# Load and tokenize data
print("\nLoading and tokenizing data...")
try:
    all_tokens = []
    num_records = 0
    max_records = 100  # Limit for testing
    
    # Read JSONL file
    with open('../pmc_articles.jsonl', 'r') as f:
        for line in f:
            if num_records >= max_records:
                break
            try:
                record = json.loads(line)
                # Extract text - adjust key based on your JSON structure
                text = record.get('text', '') or record.get('content', '') or record.get('abstract', '')
                
                if text:
                    # Tokenize with SentencePiece
                    tokens = spp.encode(text)
                    all_tokens.extend(tokens)
                    num_records += 1
                    
                    if num_records % 20 == 0:
                        print(f"  Processed {num_records} records, {len(all_tokens)} tokens so far")
            except Exception as e:
                continue
    
    print(f"✓ Loaded {num_records} records, {len(all_tokens)} total tokens")
    
    # Convert to tensor
    all_tokens = torch.tensor(all_tokens, dtype=torch.long, device=device)
    print(f"✓ Tokens tensor shape: {all_tokens.shape}")
    
except Exception as e:
    print(f"✗ Error loading data: {e}")
    raise

# Create batches from token sequence
print("\nCreating batches...")
batch_size = 4
seq_length = 128

train_data = []
for i in range(0, len(all_tokens) - seq_length, batch_size * seq_length):
    batch_tokens = []
    for j in range(batch_size):
        start = i + j * seq_length
        end = start + seq_length + 1
        
        if end <= len(all_tokens):
            seq = all_tokens[start:end]
            batch_tokens.append(seq)
    
    if len(batch_tokens) == batch_size:
        batch = torch.stack(batch_tokens)
        train_data.append(batch)

print(f"✓ Created {len(train_data)} batches of shape (batch_size=4, seq_length=129)")

# Training loop
print("\n" + "="*50)
print("Starting Training with Real Tokenized Data")
print("="*50)

model.train()
num_epochs = 1

try:
    for epoch in range(num_epochs):
        total_loss = 0
        for batch_idx, batch_tokens in enumerate(train_data):
            input_ids = batch_tokens[:, :-1]
            targets = batch_tokens[:, 1:]
            
            # Forward pass
            logits, loss = model(input_ids, targets)
            
            # Backward pass
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            loss_val = loss.item()
            total_loss += loss_val
            
            if (batch_idx + 1) % 5 == 0 or (batch_idx + 1) == len(train_data):
                print(f"Batch {batch_idx+1}/{len(train_data)} | Loss: {loss_val:.4f}")
        
        avg_loss = total_loss / len(train_data)
        print(f"\nEpoch {epoch+1} | Avg Loss: {avg_loss:.4f}")
    
    print("\n✓ Training completed successfully with real tokenized data!")

except Exception as e:
    print(f"\n✗ Error: {e}")
    import traceback
    traceback.print_exc()

Using device: cuda
number of parameters: 1008.85M
num decayed parameter tensors: 194, with 1,009,358,080 parameters
num non-decayed parameter tensors: 386, with 801,280 parameters
using fused AdamW: True
✓ Optimizer configured

Loading and tokenizing data...
  Processed 20 records, 255470 tokens so far
  Processed 40 records, 507308 tokens so far
  Processed 60 records, 729474 tokens so far
  Processed 80 records, 960001 tokens so far
  Processed 100 records, 1236101 tokens so far
✓ Loaded 100 records, 1236101 total tokens
✓ Tokens tensor shape: torch.Size([1236101])

Creating batches...
✓ Created 2414 batches of shape (batch_size=4, seq_length=129)

Starting Training with Real Tokenized Data

✗ Error: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.



Traceback (most recent call last):
  File "C:\Users\vaibh\AppData\Local\Temp\ipykernel_25280\505073623.py", line 103, in <module>
    optimizer.step()
    ~~~~~~~~~~~~~~^^
  File "c:\Users\vaibh\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\optimizer.py", line 493, in wrapper
    out = func(*args, **kwargs)
  File "c:\Users\vaibh\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\optimizer.py", line 91, in _use_grad
    ret = func(self, *args, **kwargs)
  File "c:\Users\vaibh\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\adamw.py", line 232, in step
    has_complex = self._init_group(
        group,
    ...<6 lines>...
        state_steps,
    )
  File "c:\Users\vaibh\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\adamw.py", line 175, in _init_group
    state["exp_avg_sq"] = torch.zeros_like(
                          ~~~~~~~~~~~~~~~~^
        p, memory_format=torch.preserve_format
        ^^^^^^^

In [13]:
# Generate Samples from Trained Model with Custom Prompts
print("\n" + "="*50)
print("Text Generation with Custom Prompts")
print("="*50)

model.eval()

# Generation parameters
max_new_tokens = 100
temperature = 0.7
top_k = 40

# Define custom prompts
prompts = [
    "The study shows that",
    "In medical research,",
    "The results indicate that"
]

try:
    with torch.no_grad():
        for prompt_id, prompt_text in enumerate(prompts):
            print(f"\n--- Prompt {prompt_id + 1}: \"{prompt_text}\" ---")
            
            # Tokenize the prompt
            seed_tokens_list = spp.encode(prompt_text)
            seed_tokens = torch.tensor(seed_tokens_list, dtype=torch.long, device=device).unsqueeze(0)
            
            print(f"Seed tokens ({len(seed_tokens_list)}): {seed_tokens_list}")
            
            # Generate new tokens
            generated = model.generate(
                seed_tokens,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_k=top_k
            )
            
            # Decode tokens to text
            token_list = generated[0].cpu().numpy().tolist()
            decoded_text = spp.decode(token_list)
            
            print(f"\nGenerated text:\n{decoded_text}\n")
    
    print("="*50)
    print("✓ Text generation completed!")
    print("="*50)

except Exception as e:
    print(f"✗ Error generating text: {e}")
    import traceback
    traceback.print_exc()

# INTERACTIVE MODE - Uncomment to use custom input
"""
print("\n" + "="*50)
print("Interactive Mode - Enter your own prompt")
print("="*50)

user_prompt = input("\nEnter a prompt: ").strip()

if user_prompt:
    with torch.no_grad():
        seed_tokens_list = spp.encode(user_prompt)
        seed_tokens = torch.tensor(seed_tokens_list, dtype=torch.long, device=device).unsqueeze(0)
        
        generated = model.generate(
            seed_tokens,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k
        )
        
        token_list = generated[0].cpu().numpy().tolist()
        decoded_text = spp.decode(token_list)
        
        print(f"\nYour prompt: \"{user_prompt}\"")
        print(f"\nGenerated continuation:\n{decoded_text}")
"""


Text Generation with Custom Prompts

--- Prompt 1: "The study shows that" ---
Seed tokens (4): [339, 918, 2247, 338]

Generated text:
The study shows that the same model in the this time the blood and an A and the number to the increase in the the blood the same it in the number. In the results of the a most the patients in the present in the blood the high in the number in the brain in the data is the samples in the control to the first on the three the high in the data in the be# be the data in the ability to the number, well to a effect in the this the three the study in the used


--- Prompt 2: "In medical research," ---
Seed tokens (4): [455, 1933, 1015, 49302]

Generated text:
In medical research, the EC, the A., Liom-R. *N-S., K.R., Zhang J. J., Z.. **R., S.. **E.. **j.A.H.,sed of the study.B., B., S. J.. **T., Wang**. *20**. Med. *J. *F., K. *N., Zh.* (20.L., Al.* (2019. DOI: 10.1016/j.S., K.A


--- Prompt 3: "The results indicate that" ---
Seed tokens (4): [339, 1706, 4863, 3

'\nprint("\n" + "="*50)\nprint("Interactive Mode - Enter your own prompt")\nprint("="*50)\n\nuser_prompt = input("\nEnter a prompt: ").strip()\n\nif user_prompt:\n    with torch.no_grad():\n        seed_tokens_list = spp.encode(user_prompt)\n        seed_tokens = torch.tensor(seed_tokens_list, dtype=torch.long, device=device).unsqueeze(0)\n\n        generated = model.generate(\n            seed_tokens,\n            max_new_tokens=max_new_tokens,\n            temperature=temperature,\n            top_k=top_k\n        )\n\n        token_list = generated[0].cpu().numpy().tolist()\n        decoded_text = spp.decode(token_list)\n\n        print(f"\nYour prompt: "{user_prompt}"")\n        print(f"\nGenerated continuation:\n{decoded_text}")\n'

In [ ]:
# Quick model instantiation to verify param count
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = config()
model = GPT(cfg)
model = model.to(device)
model_size = sum(p.numel() for p in model.parameters())
print(f"\nModel size: {model_size/1e6:.2f}M parameters")
print(f"Config: n_layer={cfg.n_layer}, n_head={cfg.n_head}, n_kv_heads={cfg.n_kv_heads}, n_embd={cfg.n_embd}")
print(f"Architecture: GQA + RoPE + SwiGLU + RMSNorm (Pre-LN) + Weight Tying")